# 03. Model Training
Trains all three models: **Baseline (Linear Regression)**, **XGBoost (Tuned)**, and **Deep Learning (MLP)**.

> **Important:** `SalePrice` in this dataset is already log-transformed. The models train directly on these values — no extra `np.log1p()` is applied.

In [ ]:
import sys, os

# --- Colab / Windows path bootstrap ---
if 'google.colab' in sys.modules:
    %cd /content/house-pridiction
else:
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    os.chdir(project_root)

import pandas as pd
import numpy as np
import json
from src.models.baseline import BaselineModel
from src.models.xgboost_model import XGBoostModel
from src.models.deep_learning import DeepLearningModel
from src.config import PathConfig, ModelConfig

print(f"Project root  : {os.getcwd()}")
print(f"Models dir    : {PathConfig.MODELS_DIR}")
print(f"Processed data: {PathConfig.PROCESSED_DATA}")

## 1. Load Processed Data

In [ ]:
if not os.path.exists(PathConfig.PROCESSED_DATA):
    raise FileNotFoundError(f"❌ Processed data not found: {PathConfig.PROCESSED_DATA}\nRun notebook 02 or: python -m src.data.make_dataset")

data = pd.read_csv(PathConfig.PROCESSED_DATA)
X = data.drop(columns=[ModelConfig.TARGET_COL])

# SalePrice is already log-transformed in this dataset — no extra log1p needed
y = data[ModelConfig.TARGET_COL]

print(f"✅ Data loaded: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Target range  : {y.min():.2f} – {y.max():.2f} (log scale)")
print(f"Dollar range  : ${np.expm1(y.min()):,.0f} – ${np.expm1(y.max()):,.0f}")

## 2. Train Baseline Model (Linear Regression)

In [ ]:
print("--- Training Baseline (Linear Regression) ---")
try:
    baseline = BaselineModel()
    baseline.train(X, y)
    baseline.save(os.path.join(PathConfig.MODELS_DIR, 'baseline_lr.joblib'))
    print("✅ Baseline model saved: models/baseline_lr.joblib")
except Exception as e:
    print(f"❌ Baseline training failed: {e}")

## 3. Train XGBoost Model (with Hyperparameter Tuning)

In [ ]:
print("--- Training XGBoost (Hyperparameter Tuning) ---")
try:
    xgb = XGBoostModel()
    xgb.train(X, y)
    # Saves both model.pkl and preprocessing.pkl per documentation
    xgb.save(os.path.join(PathConfig.MODELS_DIR, 'model.pkl'))

    # Save metadata
    metadata = {
        "model_name": "Elite_XGBoost_V1",
        "version": "1.0",
        "architecture": "XGBRegressor",
        "best_params": xgb.best_params,
        "target": ModelConfig.TARGET_COL,
        "log_transform": ModelConfig.LOG_TRANSFORM,
        "note": "Dataset SalePrice is pre-log-transformed. No extra log1p applied."
    }
    with open(os.path.join(PathConfig.MODELS_DIR, 'model_metadata.json'), 'w') as f:
        json.dump(metadata, f, indent=4)

    print("✅ XGBoost model saved: models/model.pkl")
    print(f"   Best params: {xgb.best_params}")
except Exception as e:
    print(f"❌ XGBoost training failed: {e}")

## 4. Train Deep Learning Model (MLP Neural Network)

In [ ]:
print("--- Training Deep Learning (MLP - 10 Epochs) ---")
try:
    dl = DeepLearningModel()
    dl.train(X, y)
    dl.save(os.path.join(PathConfig.MODELS_DIR, 'deep_learning_model.pth'))
    print("✅ Deep Learning model saved: models/deep_learning_model.pth")
except Exception as e:
    print(f"❌ Deep Learning training failed: {e}")

## 5. Verify Saved Artifacts

In [ ]:
artifacts = [
    ('Baseline model',    'baseline_lr.joblib'),
    ('XGBoost model',     'model.pkl'),
    ('Preprocessor',      'preprocessing.pkl'),
    ('Deep Learning',     'deep_learning_model.pth'),
    ('Metadata',          'model_metadata.json'),
]

print(f"{'Artifact':<20} | {'File':<30} | Status")
print("-" * 65)
for name, fname in artifacts:
    path = os.path.join(PathConfig.MODELS_DIR, fname)
    status = f"✅ {os.path.getsize(path):,} bytes" if os.path.exists(path) else "❌ Missing"
    print(f"{name:<20} | {fname:<30} | {status}")